# 10.3 Mobile Deployment — Apply

## Objective

Build and analyze ONNX models optimized for mobile deployment: constrained memory,
single-threaded inference, INT8 quantization, and battery-aware strategies.

**Prerequisites:** `pip install onnx onnxruntime numpy`

## Table of Contents
1. [Setup](#setup)
2. [Exercise 1 — Mobile-Optimized Session Configuration](#ex1)
3. [Exercise 2 — Model Size Analysis](#ex2)
4. [Exercise 3 — INT8 Quantization for Mobile](#ex3)
5. [Exercise 4 — Preprocessing Parity](#ex4)
6. [Exercise 5 — Battery-Aware Inference Strategies](#ex5)
7. [Exercise 6 — Model Packaging with Metadata](#ex6)
8. [Exercise 7 — Latency Measurement at Various Sizes](#ex7)
9. [Challenge — Mobile Deployment Package Generator](#challenge)
10. [Summary](#summary)

In [ ]:
!pip install onnx onnxruntime numpy -q

<a id='setup'></a>
## Setup

We create a small CNN suitable for mobile inference (inspired by MobileNet-style architecture)
using `onnx.helper`. This gives us a model to profile, quantize, and package.

In [ ]:
import os
import time
import json
import struct
import hashlib
import numpy as np
from typing import Any, Dict, List, Optional, Tuple

import onnx
from onnx import helper, TensorProto, numpy_helper
import onnxruntime as ort

np.random.seed(42)

def make_conv_relu_block(prefix: str, in_ch: int, out_ch: int, kernel: int = 3):
    """Create Conv + Relu nodes with random initializers."""
    W = numpy_helper.from_array(
        np.random.randn(out_ch, in_ch, kernel, kernel).astype(np.float32) * 0.1,
        name=f"{prefix}_W"
    )
    B = numpy_helper.from_array(
        np.zeros(out_ch, dtype=np.float32),
        name=f"{prefix}_B"
    )
    conv = helper.make_node(
        "Conv", [f"{prefix}_input", f"{prefix}_W", f"{prefix}_B"],
        [f"{prefix}_conv_out"],
        kernel_shape=[kernel, kernel], pads=[1, 1, 1, 1]
    )
    relu = helper.make_node("Relu", [f"{prefix}_conv_out"], [f"{prefix}_output"])
    return [conv, relu], [W, B]


def build_mobile_cnn(num_classes: int = 10) -> onnx.ModelProto:
    """Build a small CNN: Conv(3->16) -> Conv(16->32) -> GlobalAvgPool -> FC."""
    nodes, inits = [], []

    # Block 1: 3 -> 16
    b1_nodes, b1_inits = make_conv_relu_block("b1", 3, 16)
    nodes.extend(b1_nodes)
    inits.extend(b1_inits)

    # Block 2: 16 -> 32
    b2_nodes, b2_inits = make_conv_relu_block("b2", 16, 32)
    nodes.extend(b2_nodes)
    inits.extend(b2_inits)

    # Rename intermediate connections
    nodes[2].input[0] = "b1_output"  # b2 input = b1 output

    # Global Average Pooling
    gap = helper.make_node("GlobalAveragePool", ["b2_output"], ["gap_out"])
    flatten = helper.make_node("Flatten", ["gap_out"], ["flat_out"], axis=1)
    nodes.extend([gap, flatten])

    # FC layer: 32 -> num_classes
    fc_W = numpy_helper.from_array(
        np.random.randn(32, num_classes).astype(np.float32) * 0.1, name="fc_W"
    )
    fc_B = numpy_helper.from_array(np.zeros(num_classes, dtype=np.float32), name="fc_B")
    matmul = helper.make_node("MatMul", ["flat_out", "fc_W"], ["mm_out"])
    add = helper.make_node("Add", ["mm_out", "fc_B"], ["logits"])
    nodes.extend([matmul, add])
    inits.extend([fc_W, fc_B])

    # Rename first input
    nodes[0].input[0] = "image"

    X = helper.make_tensor_value_info("image", TensorProto.FLOAT, [1, 3, 32, 32])
    Y = helper.make_tensor_value_info("logits", TensorProto.FLOAT, [1, num_classes])

    graph = helper.make_graph(nodes, "mobile_cnn", [X], [Y], inits)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    model.ir_version = 8
    onnx.checker.check_model(model)
    return model


mobile_model = build_mobile_cnn()
MODEL_PATH = "/tmp/mobile_cnn.onnx"
onnx.save(mobile_model, MODEL_PATH)
print(f"Mobile CNN saved: {os.path.getsize(MODEL_PATH)} bytes")
print(f"Nodes: {len(mobile_model.graph.node)}, Initializers: {len(mobile_model.graph.initializer)}")

<a id='ex1'></a>
## Exercise 1 — Mobile-Optimized Session Configuration

Mobile devices have limited resources. Key constraints:

- **Single thread**: avoid contention with UI thread
- **Disable memory arena**: reduce peak RSS on low-RAM devices
- **Sequential execution**: predictable memory usage

Memory usage on mobile:

$$\text{RSS}_{\text{peak}} \approx \text{model\_size} + \text{max\_activation\_tensor} + \text{runtime\_overhead}$$

In [ ]:
def build_mobile_session(
    model_path: str,
    memory_budget_mb: float = 50.0,
) -> Tuple[ort.InferenceSession, Dict[str, Any]]:
    """Build a session optimized for mobile constraints."""
    so = ort.SessionOptions()

    # Single thread — share CPU with UI/system
    so.intra_op_num_threads = 1
    so.inter_op_num_threads = 1

    # Sequential execution — predictable memory, no thread pool overhead
    so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

    # Disable memory arena — reduces peak memory at cost of alloc speed
    so.enable_cpu_mem_arena = False
    so.enable_mem_pattern = False

    # Enable all graph optimizations — fuse ops to reduce kernel launch overhead
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    session = ort.InferenceSession(
        model_path, sess_options=so, providers=["CPUExecutionProvider"]
    )

    model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
    config = {
        "threads": 1,
        "execution_mode": "sequential",
        "mem_arena": False,
        "model_size_mb": round(model_size_mb, 3),
        "within_budget": model_size_mb < memory_budget_mb,
        "budget_mb": memory_budget_mb,
    }
    return session, config


mobile_session, mobile_config = build_mobile_session(MODEL_PATH)
print("Mobile session config:")
print(json.dumps(mobile_config, indent=2))

# Verify inference works
dummy = np.random.randn(1, 3, 32, 32).astype(np.float32)
output = mobile_session.run(None, {"image": dummy})[0]
assert output.shape == (1, 10)
assert mobile_config["within_budget"]
print(f"\nInference OK — output shape: {output.shape}")

<a id='ex2'></a>
## Exercise 2 — Model Size Analysis

On mobile, download size and on-device storage matter. We analyze where bytes go.

For a neural network, model size is dominated by parameters:

$$\text{size}_{\text{bytes}} = \sum_{\text{layer}} \text{params}_{\text{layer}} \times \text{bytes\_per\_param}$$

For FP32: 4 bytes/param. INT8: 1 byte/param → **4× compression**.

In [ ]:
def analyze_model_size(model_path: str) -> Dict[str, Any]:
    """Decompose model size into parameters, structure, and metadata."""
    model = onnx.load(model_path)
    total_file_size = os.path.getsize(model_path)

    # Analyze initializers (weights)
    param_details = []
    total_params = 0
    total_param_bytes = 0

    dtype_sizes = {
        TensorProto.FLOAT: 4,
        TensorProto.FLOAT16: 2,
        TensorProto.INT8: 1,
        TensorProto.UINT8: 1,
        TensorProto.INT32: 4,
    }

    for init in model.graph.initializer:
        shape = list(init.dims)
        n_params = int(np.prod(shape)) if shape else 1
        elem_size = dtype_sizes.get(init.data_type, 4)
        byte_size = n_params * elem_size

        param_details.append({
            "name": init.name,
            "shape": shape,
            "params": n_params,
            "dtype_bytes": elem_size,
            "total_bytes": byte_size,
        })
        total_params += n_params
        total_param_bytes += byte_size

    overhead = total_file_size - total_param_bytes

    return {
        "file_size_bytes": total_file_size,
        "file_size_kb": round(total_file_size / 1024, 2),
        "total_params": total_params,
        "param_bytes": total_param_bytes,
        "overhead_bytes": overhead,
        "param_pct": round(100 * total_param_bytes / total_file_size, 1),
        "layers": param_details,
        "potential_int8_size_kb": round((total_params + overhead) / 1024, 2),
    }


analysis = analyze_model_size(MODEL_PATH)
print(f"File size: {analysis['file_size_kb']} KB")
print(f"Total parameters: {analysis['total_params']:,}")
print(f"Parameters account for {analysis['param_pct']}% of file")
print(f"Potential INT8 size: ~{analysis['potential_int8_size_kb']} KB")
print("\nPer-layer breakdown:")
for layer in analysis["layers"]:
    print(f"  {layer['name']:20s} shape={str(layer['shape']):15s} params={layer['params']:>8,}")

assert analysis["total_params"] > 0
assert analysis["param_pct"] > 50  # params should dominate

<a id='ex3'></a>
## Exercise 3 — INT8 Quantization for Mobile

Dynamic quantization converts FP32 weights to INT8 at load time.

Quantization formula (affine):

$$q = \text{round}\left(\frac{x}{s}\right) + z$$

where $s$ is the scale and $z$ is the zero-point. Dequantization:

$$x \approx s \cdot (q - z)$$

Benefits on mobile:
- 4× smaller model file
- Faster inference on ARM NEON (INT8 GEMM)
- Lower power consumption

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

QUANT_PATH = "/tmp/mobile_cnn_int8.onnx"

quantize_dynamic(
    model_input=MODEL_PATH,
    model_output=QUANT_PATH,
    weight_type=QuantType.QUInt8,
)

# Compare sizes
fp32_size = os.path.getsize(MODEL_PATH)
int8_size = os.path.getsize(QUANT_PATH)
compression = fp32_size / int8_size

print(f"FP32 model: {fp32_size:,} bytes")
print(f"INT8 model: {int8_size:,} bytes")
print(f"Compression ratio: {compression:.2f}x")

# Verify quantized model produces similar outputs
quant_session, _ = build_mobile_session(QUANT_PATH)
fp32_out = mobile_session.run(None, {"image": dummy})[0]
int8_out = quant_session.run(None, {"image": dummy})[0]

max_diff = float(np.max(np.abs(fp32_out - int8_out)))
print(f"\nMax absolute difference (FP32 vs INT8): {max_diff:.6f}")
assert max_diff < 1.0, "Quantization error too large"

# Both should agree on top prediction for reasonable inputs
assert np.argmax(fp32_out) == np.argmax(int8_out), "Top prediction differs!"
print(f"Both models agree: class {np.argmax(fp32_out)}")

<a id='ex4'></a>
## Exercise 4 — Preprocessing Parity

Mobile preprocessing must produce **identical** tensors to server-side preprocessing.
Differences cause accuracy drops. We verify parity between a "mobile" (integer-only)
and "server" (float) pipeline.

Image normalization:

$$x_{\text{norm}} = \frac{x_{\text{uint8}} / 255.0 - \mu}{\sigma}$$

In [ ]:
def server_preprocess(
    image_uint8: np.ndarray,
    target_size: Tuple[int, int] = (32, 32),
    mean: Tuple[float, ...] = (0.485, 0.456, 0.406),
    std: Tuple[float, ...] = (0.229, 0.224, 0.225),
) -> np.ndarray:
    """Server-side preprocessing (float64 intermediate for precision)."""
    # Simulate resize via interpolation (simple nearest-neighbor for demo)
    h, w = image_uint8.shape[:2]
    th, tw = target_size
    row_idx = (np.arange(th) * h / th).astype(int)
    col_idx = (np.arange(tw) * w / tw).astype(int)
    resized = image_uint8[row_idx][:, col_idx]

    # Normalize
    x = resized.astype(np.float64) / 255.0
    mean_arr = np.array(mean, dtype=np.float64).reshape(1, 1, 3)
    std_arr = np.array(std, dtype=np.float64).reshape(1, 1, 3)
    x = (x - mean_arr) / std_arr

    # HWC -> CHW, add batch dim
    x = np.transpose(x, (2, 0, 1))[np.newaxis, ...]
    return x.astype(np.float32)


def mobile_preprocess(
    image_uint8: np.ndarray,
    target_size: Tuple[int, int] = (32, 32),
    mean: Tuple[float, ...] = (0.485, 0.456, 0.406),
    std: Tuple[float, ...] = (0.229, 0.224, 0.225),
) -> np.ndarray:
    """Mobile preprocessing (float32 throughout, same algorithm)."""
    h, w = image_uint8.shape[:2]
    th, tw = target_size
    row_idx = (np.arange(th) * h / th).astype(int)
    col_idx = (np.arange(tw) * w / tw).astype(int)
    resized = image_uint8[row_idx][:, col_idx]

    # Mobile: FP32 throughout
    x = resized.astype(np.float32) / 255.0
    mean_arr = np.array(mean, dtype=np.float32).reshape(1, 1, 3)
    std_arr = np.array(std, dtype=np.float32).reshape(1, 1, 3)
    x = (x - mean_arr) / std_arr

    x = np.transpose(x, (2, 0, 1))[np.newaxis, ...]
    return x


# Create a synthetic image (64x64 RGB)
synthetic_image = np.random.randint(0, 256, size=(64, 64, 3), dtype=np.uint8)

server_tensor = server_preprocess(synthetic_image)
mobile_tensor = mobile_preprocess(synthetic_image)

max_diff = float(np.max(np.abs(server_tensor - mobile_tensor)))
print(f"Max preprocessing difference (server vs mobile): {max_diff:.2e}")
assert max_diff < 1e-5, "Preprocessing parity violation!"
print("Preprocessing parity verified.")

# Verify same inference result
out_server = mobile_session.run(None, {"image": server_tensor})[0]
out_mobile = mobile_session.run(None, {"image": mobile_tensor})[0]
assert np.allclose(out_server, out_mobile, atol=1e-5)
print("Inference parity confirmed.")

<a id='ex5'></a>
## Exercise 5 — Battery-Aware Inference Strategies

Mobile inference must adapt to battery state. Strategy:

| Battery Level | Strategy | Quality | Power |
|:---:|:---:|:---:|:---:|
| > 50% | Full model (FP32) | High | High |
| 20–50% | Quantized (INT8) | Medium | Low |
| < 20% | Skip / cache results | Minimal | Minimal |

Energy per inference:

$$E_{\text{infer}} \approx P_{\text{cpu}} \times t_{\text{latency}} = V \cdot I \cdot t$$

In [ ]:
class BatteryAwareInference:
    """Adapts model quality based on simulated battery level."""

    def __init__(
        self,
        fp32_session: ort.InferenceSession,
        int8_session: ort.InferenceSession,
        input_name: str = "image",
    ):
        self.fp32_session = fp32_session
        self.int8_session = int8_session
        self.input_name = input_name
        self._cache: Dict[str, np.ndarray] = {}
        self.stats = {"fp32_calls": 0, "int8_calls": 0, "cache_hits": 0, "skipped": 0}

    def _cache_key(self, tensor: np.ndarray) -> str:
        return hashlib.md5(tensor.tobytes()).hexdigest()

    def infer(self, tensor: np.ndarray, battery_pct: float) -> Optional[np.ndarray]:
        """Run inference with battery-adaptive quality."""
        key = self._cache_key(tensor)

        # Low battery: use cache or skip
        if battery_pct < 20:
            if key in self._cache:
                self.stats["cache_hits"] += 1
                return self._cache[key]
            self.stats["skipped"] += 1
            return None

        # Medium battery: quantized model
        if battery_pct < 50:
            self.stats["int8_calls"] += 1
            result = self.int8_session.run(None, {self.input_name: tensor})[0]
        else:
            # High battery: full precision
            self.stats["fp32_calls"] += 1
            result = self.fp32_session.run(None, {self.input_name: tensor})[0]

        self._cache[key] = result
        return result


# Simulate battery drain
battery_inference = BatteryAwareInference(mobile_session, quant_session)

test_images = [np.random.randn(1, 3, 32, 32).astype(np.float32) for _ in range(5)]

battery_levels = [80, 60, 40, 25, 10]
for img, battery in zip(test_images, battery_levels):
    result = battery_inference.infer(img, battery)
    status = "result" if result is not None else "skipped"
    print(f"Battery {battery}%: {status}")

# Repeat last image at low battery — should hit cache
result = battery_inference.infer(test_images[3], battery_pct=5)
assert result is not None  # cached from when battery was 25%

print(f"\nStats: {json.dumps(battery_inference.stats, indent=2)}")
assert battery_inference.stats["fp32_calls"] == 2
assert battery_inference.stats["int8_calls"] == 2
assert battery_inference.stats["cache_hits"] == 1

<a id='ex6'></a>
## Exercise 6 — Model Packaging with Metadata

Mobile deployments need versioning, compatibility checks, and integrity verification.
We create a packaging format that bundles model + metadata.

In [ ]:
def package_model_for_mobile(
    model_path: str,
    version: str = "1.0.0",
    min_sdk_version: str = "1.14.0",
    target_devices: List[str] = None,
    input_spec: Optional[Dict] = None,
) -> Dict[str, Any]:
    """Create a deployment package manifest for mobile."""
    if target_devices is None:
        target_devices = ["ios_arm64", "android_arm64"]

    # Compute file hash for integrity
    with open(model_path, "rb") as f:
        model_bytes = f.read()
    sha256 = hashlib.sha256(model_bytes).hexdigest()

    # Introspect model
    model = onnx.load(model_path)
    inputs = []
    for inp in model.graph.input:
        shape = [d.dim_value if d.dim_value > 0 else "dynamic"
                 for d in inp.type.tensor_type.shape.dim]
        inputs.append({"name": inp.name, "shape": shape})

    package = {
        "format_version": "1.0",
        "model": {
            "filename": os.path.basename(model_path),
            "size_bytes": len(model_bytes),
            "sha256": sha256,
            "opset_version": model.opset_import[0].version,
            "ir_version": model.ir_version,
        },
        "deployment": {
            "version": version,
            "min_ort_version": min_sdk_version,
            "target_devices": target_devices,
            "quantized": "Quantize" in str([n.op_type for n in model.graph.node]),
        },
        "interface": {
            "inputs": inputs or input_spec,
            "preprocessing": {
                "normalize": True,
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225],
                "input_format": "NCHW",
                "input_dtype": "float32",
            },
        },
    }
    return package


# Package both FP32 and INT8 models
fp32_pkg = package_model_for_mobile(MODEL_PATH, version="1.0.0")
int8_pkg = package_model_for_mobile(QUANT_PATH, version="1.0.0-int8")

print("FP32 Package:")
print(json.dumps(fp32_pkg, indent=2))

assert fp32_pkg["model"]["sha256"] != int8_pkg["model"]["sha256"]
assert fp32_pkg["model"]["size_bytes"] > int8_pkg["model"]["size_bytes"]
print(f"\nSize reduction: {fp32_pkg['model']['size_bytes']} -> {int8_pkg['model']['size_bytes']} bytes")

<a id='ex7'></a>
## Exercise 7 — Latency Measurement at Various Sizes

We compare inference latency across model variants to inform deployment decisions.

On ARM devices, expected scaling:

$$t_{\text{INT8}} \approx \frac{t_{\text{FP32}}}{2 \text{ to } 4}$$

due to NEON INT8 GEMM throughput advantages.

In [ ]:
def benchmark_latency(
    session: ort.InferenceSession,
    input_name: str,
    input_shape: List[int],
    num_warmup: int = 10,
    num_runs: int = 100,
) -> Dict[str, float]:
    """Measure inference latency with warmup."""
    tensor = np.random.randn(*input_shape).astype(np.float32)

    # Warmup
    for _ in range(num_warmup):
        session.run(None, {input_name: tensor})

    # Measure
    latencies = []
    for _ in range(num_runs):
        start = time.perf_counter()
        session.run(None, {input_name: tensor})
        latencies.append((time.perf_counter() - start) * 1000)

    lat = np.array(latencies)
    return {
        "mean_ms": round(float(np.mean(lat)), 3),
        "std_ms": round(float(np.std(lat)), 3),
        "p50_ms": round(float(np.percentile(lat, 50)), 3),
        "p95_ms": round(float(np.percentile(lat, 95)), 3),
        "min_ms": round(float(np.min(lat)), 3),
        "max_ms": round(float(np.max(lat)), 3),
    }


# Benchmark FP32
fp32_latency = benchmark_latency(mobile_session, "image", [1, 3, 32, 32])
print("FP32 Latency:", json.dumps(fp32_latency, indent=2))

# Benchmark INT8
int8_latency = benchmark_latency(quant_session, "image", [1, 3, 32, 32])
print("\nINT8 Latency:", json.dumps(int8_latency, indent=2))

# Summary
speedup = fp32_latency["mean_ms"] / max(int8_latency["mean_ms"], 1e-6)
print(f"\nSpeedup (INT8 vs FP32): {speedup:.2f}x")
print(f"FP32 achieves ~{1000/fp32_latency['mean_ms']:.0f} inferences/sec")
print(f"INT8 achieves ~{1000/int8_latency['mean_ms']:.0f} inferences/sec")

<a id='challenge'></a>
## Challenge — Mobile Deployment Package Generator

Create a complete mobile deployment pipeline that:
1. Takes any ONNX model
2. Quantizes it
3. Benchmarks both variants
4. Generates a deployment package with recommendation

In [ ]:
def generate_mobile_deployment(
    model_path: str,
    version: str = "1.0.0",
    max_latency_ms: float = 100.0,
    max_size_mb: float = 10.0,
) -> Dict[str, Any]:
    """Complete mobile deployment package generator."""
    # Step 1: Analyze original model
    size_analysis = analyze_model_size(model_path)

    # Step 2: Quantize
    quant_path = model_path.replace(".onnx", "_mobile_int8.onnx")
    quantize_dynamic(model_input=model_path, model_output=quant_path, weight_type=QuantType.QUInt8)

    # Step 3: Build sessions
    fp32_sess, _ = build_mobile_session(model_path)
    int8_sess, _ = build_mobile_session(quant_path)

    # Step 4: Benchmark
    inp = fp32_sess.get_inputs()[0]
    shape = [1 if isinstance(d, str) else d for d in inp.shape]

    fp32_bench = benchmark_latency(fp32_sess, inp.name, shape, num_runs=50)
    int8_bench = benchmark_latency(int8_sess, inp.name, shape, num_runs=50)

    # Step 5: Make recommendation
    fp32_ok = (
        fp32_bench["p95_ms"] <= max_latency_ms
        and size_analysis["file_size_kb"] / 1024 <= max_size_mb
    )
    int8_size_kb = os.path.getsize(quant_path) / 1024
    int8_ok = (
        int8_bench["p95_ms"] <= max_latency_ms
        and int8_size_kb / 1024 <= max_size_mb
    )

    if int8_ok:
        recommendation = "int8"
        reason = "Meets latency/size constraints with smaller footprint"
    elif fp32_ok:
        recommendation = "fp32"
        reason = "INT8 not available; FP32 meets constraints"
    else:
        recommendation = "needs_optimization"
        reason = "Neither variant meets constraints; consider pruning or architecture change"

    return {
        "version": version,
        "analysis": {
            "fp32_size_kb": size_analysis["file_size_kb"],
            "int8_size_kb": round(int8_size_kb, 2),
            "compression_ratio": round(size_analysis["file_size_kb"] / int8_size_kb, 2),
            "total_params": size_analysis["total_params"],
        },
        "benchmarks": {
            "fp32": fp32_bench,
            "int8": int8_bench,
        },
        "constraints": {
            "max_latency_ms": max_latency_ms,
            "max_size_mb": max_size_mb,
        },
        "recommendation": {
            "variant": recommendation,
            "reason": reason,
            "model_path": quant_path if recommendation == "int8" else model_path,
        },
        "package": package_model_for_mobile(
            quant_path if recommendation == "int8" else model_path,
            version=version,
        ),
    }


deployment = generate_mobile_deployment(MODEL_PATH, max_latency_ms=50.0)
print("Mobile Deployment Report:")
print(json.dumps(deployment, indent=2))

assert deployment["recommendation"]["variant"] in ["int8", "fp32", "needs_optimization"]
assert deployment["analysis"]["compression_ratio"] >= 1.0
print(f"\nRecommendation: {deployment['recommendation']['variant']}")
print(f"Reason: {deployment['recommendation']['reason']}")

<a id='summary'></a>
## Summary

| Component | Mobile Consideration |
|-----------|---------------------|
| **Session Config** | Single thread, no memory arena, sequential execution |
| **Size Analysis** | Parameter count drives download/storage budget |
| **Quantization** | INT8 gives ~4× compression, 1-4× speedup on ARM |
| **Preprocessing** | Must match server exactly for accuracy parity |
| **Battery Awareness** | Adapt quality to power state |
| **Packaging** | Version, hash, compatibility metadata |
| **Latency** | Profile both variants; P95 determines UX quality |

**Key insight:** Mobile deployment is about *budgets* — latency, memory, power, and download size.
Every optimization trades off against accuracy.